# Módulo 1 — Análisis exploratorio, desbalance y pipeline de datos
**TFM · Clasificación de lesiones cutáneas (HAM10000) con CNN y Grad-CAM**
Responsable: **Adrián Santiago Arias Navarrete** · Objetivo específico: **OE1**

### Contrato de salida (lo que consumen los Módulos 2 y 3)
| Artefacto | Ruta en Drive (`TFM/data/`) | Uso |
|---|---|---|
| Imágenes redimensionadas | `ham10000_images_224_stretch.zip` | Entrenamiento y Grad-CAM |
| Metadatos | `HAM10000_metadata.csv` | Análisis de errores |
| **Split fijo** (único para los 3 módulos) | `split_v1.csv` (+ copia en el repo `data/`) | Train/val/test |
| Pesos de clase | `class_weights_v1.json` | Entrenamiento |

> El split **no se regenera** una vez publicado. Si hubiera que cambiarlo, subir `SPLIT_VERSION` en `tfm_common.py` y avisar al grupo.

In [ ]:
# @title 0. Entorno: Drive, repositorio y dependencias
# Requisitos (una sola vez por persona):
#  - En Drive: "Añadir acceso directo" de la carpeta compartida TFM a "Mi unidad".
#  - En Colab > Secretos (icono de llave): DAGSHUB_TOKEN y, si el repo es privado, GITHUB_TOKEN.
#  - Entorno de ejecución > Cambiar tipo > GPU T4.
import os, sys, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")

GITHUB_USER = "aReinoso007"
REPO_NAME = "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/TFM"   # carpeta compartida del grupo

try:
    _tok = userdata.get("GITHUB_TOKEN")
except Exception:
    _tok = None
_url = f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok else \
       f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", _url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, f"{REPO_DIR}/src")

!pip -q install "mlflow<3" dagshub kagglehub

import importlib, tfm_common as tc
importlib.reload(tc)
paths = tc.get_paths(DRIVE_ROOT)
print("Repo:", REPO_DIR, "| Drive:", paths.drive_root)

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Descarga del dataset

In [ ]:
# @title 1.1 Descarga desde Kaggle (kagglehub)
# Si pide credenciales: crear secretos KAGGLE_USERNAME y KAGGLE_KEY en Colab.
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    pass

meta_path, image_map = tc.download_ham10000()
print("Metadatos:", meta_path)
print("Imágenes encontradas:", len(image_map))
assert len(image_map) == tc.N_IMAGES_EXPECTED, "Número de imágenes inesperado"

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
FIG_DIR = paths.drive_figures / "modulo1"; FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

meta = pd.read_csv(meta_path)
meta.to_csv(paths.metadata_csv, index=False)          # copia persistente en Drive
meta.head()

## 2. Análisis exploratorio (EDA)

In [ ]:
# @title 2.1 Estructura, nulos y lesiones con varias imágenes
print(meta.shape)
print(meta.dtypes, "\n")
print("Nulos por columna:\n", meta.isna().sum(), "\n")
img_per_lesion = meta.groupby("lesion_id").size()
print(f"Lesiones únicas: {meta.lesion_id.nunique()}  |  imágenes: {len(meta)}")
print("Imágenes por lesión:\n", img_per_lesion.value_counts().sort_index())
# Varias imágenes por lesión => el split DEBE agruparse por lesion_id (evitar fuga de datos)

In [ ]:
# @title 2.2 Distribución de clases (imágenes y lesiones)
dist = pd.DataFrame({
    "imagenes": meta.dx.value_counts(),
    "lesiones": meta.drop_duplicates("lesion_id").dx.value_counts(),
}).reindex(tc.CLASSES)
dist["% imagenes"] = (100 * dist.imagenes / dist.imagenes.sum()).round(2)
dist["nombre"] = dist.index.map(tc.CLASS_NAMES_ES)
display(dist)
print(f"Ratio de desbalance (máx/mín): {dist.imagenes.max() / dist.imagenes.min():.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=dist.index, y=dist.imagenes, ax=ax, color="steelblue")
for i, v in enumerate(dist.imagenes):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Clase"); ax.set_ylabel("Nº de imágenes"); ax.set_title("Distribución de clases en HAM10000")
savefig(fig, "distribucion_clases"); plt.show()

In [ ]:
# @title 2.3 Metadatos: edad, sexo, localización y tipo de confirmación diagnóstica
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.boxplot(data=meta, x="dx", y="age", order=tc.CLASSES, ax=axes[0, 0])
axes[0, 0].set_title("Edad por clase")
pd.crosstab(meta.dx, meta.sex, normalize="index").reindex(tc.CLASSES).plot.bar(stacked=True, ax=axes[0, 1])
axes[0, 1].set_title("Sexo por clase (proporción)")
loc = pd.crosstab(meta.localization, meta.dx)[tc.CLASSES]
sns.heatmap(loc, annot=True, fmt="d", cmap="Blues", ax=axes[1, 0], cbar=False)
axes[1, 0].set_title("Localización anatómica × clase")
pd.crosstab(meta.dx, meta.dx_type, normalize="index").reindex(tc.CLASSES).plot.bar(stacked=True, ax=axes[1, 1])
axes[1, 1].set_title("Tipo de confirmación (histo / follow_up / consensus / confocal)")
fig.tight_layout(); savefig(fig, "metadatos"); plt.show()
# Nota para la memoria: la fiabilidad de la etiqueta depende de dx_type (histopatología > consenso).

In [ ]:
# @title 2.4 Propiedades de las imágenes originales y muestra por clase
from PIL import Image
sample_ids = meta.sample(min(200, len(meta)), random_state=0).image_id
sizes = pd.Series([Image.open(image_map[i]).size for i in sample_ids]).value_counts()
print("Tamaños (muestra de 200):\n", sizes)

k = 6
fig, axes = plt.subplots(len(tc.CLASSES), k, figsize=(2.2 * k, 2.0 * len(tc.CLASSES)))
for r, c in enumerate(tc.CLASSES):
    ids = meta[meta.dx == c].sample(k, random_state=1).image_id
    for j, iid in enumerate(ids):
        axes[r, j].imshow(Image.open(image_map[iid])); axes[r, j].axis("off")
    axes[r, 0].set_title(c, loc="left", fontsize=10)
fig.tight_layout(); savefig(fig, "muestras_por_clase"); plt.show()

In [ ]:
# @title 2.5 (Exploratorio) Color medio por clase — posible atajo espurio
# Si una clase difiere mucho en color/brillo medio, el modelo podría aprender ese atajo.
rows = []
for c in tc.CLASSES:
    for iid in meta[meta.dx == c].sample(min(150, (meta.dx == c).sum()), random_state=0).image_id:
        a = np.asarray(Image.open(image_map[iid]).convert("RGB").resize((64, 64)), dtype=float)
        rows.append(dict(dx=c, R=a[..., 0].mean(), G=a[..., 1].mean(), B=a[..., 2].mean(),
                         borde=np.r_[a[:6].ravel(), a[-6:].ravel()].mean()))
col = pd.DataFrame(rows)
display(col.groupby("dx")[["R", "G", "B", "borde"]].mean().reindex(tc.CLASSES).round(1))

## 3. Redimensionado
**Decisión a documentar:** `IMG_SIZE = 224` (entrada nativa de las tres arquitecturas) y `RESIZE_MODE`:
- `stretch`: 600×450 → 224×224 sin recortar (deforma ligeramente, no pierde información).
- `center_crop`: recorte cuadrado central 450×450 → 224×224 (mantiene proporción, puede cortar bordes de la lesión).

Se cambia en `tfm_common.py` (afecta a los tres módulos).

In [ ]:
# @title 3.1 Redimensionar y subir el zip a Drive (≈ 5-10 min, una sola vez)
import time
t0 = time.time()
n = tc.resize_images(image_map, paths.local_images)
print(f"{n} imágenes en {paths.local_images} ({time.time() - t0:.0f}s)")
if not paths.images_zip.exists():
    tc.zip_dir(paths.local_images, paths.images_zip)
print("Zip en Drive:", paths.images_zip, f"{paths.images_zip.stat().st_size / 1e6:.0f} MB")

## 4. Split fijo train / val / test
70/15/15, **estratificado por clase** y **agrupado por `lesion_id`** (`StratifiedGroupKFold`, 20 folds: 3 → test, 3 → val, 14 → train). Semilla fija `SPLIT_SEED = 42`. Es el único split que usan los tres módulos.

In [ ]:
# @title 4.1 Generar, verificar y guardar el split
split = tc.make_split(meta)
tc.check_no_leakage(split)

tab = pd.crosstab(split.dx, split.split).reindex(tc.CLASSES)[["train", "val", "test"]]
tab_pct = (100 * tab.div(tab.sum(axis=1), axis=0)).round(1)
display(pd.concat({"n": tab, "% por clase": tab_pct}, axis=1))
print((100 * split.split.value_counts(normalize=True)).round(2))
print("Lesiones por subconjunto:", split.groupby("split").lesion_id.nunique().to_dict())

if paths.split_csv.exists():
    prev = pd.read_csv(paths.split_csv)
    same = prev.sort_values("image_id").reset_index(drop=True).equals(split.sort_values("image_id").reset_index(drop=True))
    print("Ya existía un split en Drive. ¿Idéntico?:", same)
    assert same, "El split cambió: NO sobrescribir sin acordarlo con el grupo"
else:
    split.to_csv(paths.split_csv, index=False)
    print("Guardado:", paths.split_csv)

# Copia en el repositorio (hacer commit/push desde aquí o desde local)
os.makedirs(f"{REPO_DIR}/data", exist_ok=True)
split.to_csv(f"{REPO_DIR}/data/split_{tc.SPLIT_VERSION}.csv", index=False)

## 5. Estrategia frente al desbalance
Dos mecanismos complementarios (el Módulo 2 elige la combinación por configuración):
1. **Class weighting** en la función de pérdida (`balanced` o su versión suavizada `sqrt`).
2. **Data augmentation selectivo**: transformaciones fuertes (rotación, volteos, zoom, color) solo en clases minoritarias; ligera (volteos) en `nv`.
3. *(Opcional)* **Sobremuestreo** de minoritarias. Si se usa, los pesos de clase se recalculan sobre el train sobremuestreado para no compensar dos veces.

In [ ]:
# @title 5.1 Pesos de clase
df = tc.load_split(paths)
train_df = df[df.split == "train"]
cw = {m: tc.compute_class_weights(train_df, m) for m in ["balanced", "sqrt"]}
display(pd.DataFrame({m: {tc.CLASSES[k]: round(v, 3) for k, v in w.items()} for m, w in cw.items()}))
tc.save_json({"split_version": tc.SPLIT_VERSION, **cw}, paths.class_weights_json)
print("Guardado:", paths.class_weights_json)

In [ ]:
# @title 5.2 (Opcional) Efecto del sobremuestreo en la distribución efectiva
factors = {"akiec": 3, "bcc": 2, "bkl": 1, "df": 6, "mel": 2, "vasc": 5}   # ajustar
over = tc.oversample(train_df, factors)
display(pd.DataFrame({"original": train_df.dx.value_counts(), "sobremuestreado": over.dx.value_counts()}).reindex(tc.CLASSES))

In [ ]:
# @title 5.3 Vista previa del augmentation
import tensorflow as tf
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for r, c in enumerate(["mel", "df", "vasc", "nv"]):
    iid = train_df[train_df.dx == c].image_id.iloc[0]
    img = tf.constant(tc.load_image_uint8(str(paths.local_images / f"{iid}.jpg")), tf.float32)
    axes[r, 0].imshow(img.numpy().astype("uint8")); axes[r, 0].set_title(f"{c} original", fontsize=9)
    fn = tc.augment_light if c == "nv" else tc.augment_strong
    for j in range(1, 6):
        axes[r, j].imshow(fn(img).numpy().astype("uint8"))
    for a in axes[r]: a.axis("off")
fig.suptitle("Augmentation selectivo (fuerte en minoritarias, ligero en nv)")
fig.tight_layout(); savefig(fig, "augmentation"); plt.show()

## 6. Pipeline `tf.data`

In [ ]:
# @title 6.1 Comprobación y rendimiento del pipeline
import time
for arch in tc.ARCHS:
    ds = tc.make_dataset(train_df, arch, training=True, batch_size=32)
    x, y = next(iter(ds))
    print(f"{arch:15s} x={tuple(x.shape)} rango=[{float(tf.reduce_min(x)):.1f}, {float(tf.reduce_max(x)):.1f}] y={tuple(y.shape)}")

ds = tc.make_dataset(train_df, "resnet50", training=True, batch_size=32)
t0 = time.time(); n = sum(int(b[0].shape[0]) for b in ds); dt = time.time() - t0
print(f"Una época de train: {n} imágenes en {dt:.1f}s ({n / dt:.0f} img/s)")

## 7. Registro en MLflow (DagsHub) y resumen

In [ ]:
# @title 7.1 Registrar el EDA como run en MLflow
mlflow = tc.init_mlflow()
with mlflow.start_run(run_name="modulo1_eda_split"):
    mlflow.set_tags({"modulo": "1", "responsable": "Adrián Arias"})
    mlflow.log_params({"split_version": tc.SPLIT_VERSION, "split_seed": tc.SPLIT_SEED,
                       "img_size": tc.IMG_SIZE, "resize_mode": tc.RESIZE_MODE,
                       "n_train": int((df.split == "train").sum()), "n_val": int((df.split == "val").sum()),
                       "n_test": int((df.split == "test").sum())})
    mlflow.log_artifact(str(paths.split_csv))
    mlflow.log_artifact(str(paths.class_weights_json))
    mlflow.log_artifacts(str(FIG_DIR), artifact_path="figuras")

In [ ]:
# @title 7.2 Checklist de entrega para los Módulos 2 y 3
for p in [paths.images_zip, paths.metadata_csv, paths.split_csv, paths.class_weights_json]:
    print(("OK  " if p.exists() else "FALTA ") + str(p))
print("\nRecordatorio: commit + push de data/split_v1.csv y de este notebook.")